# Example script for Hackathon

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have limited chances to make queries.

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

import os
import esm
from tqdm.auto import tqdm

## 1. Collect Training Data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [2]:
with open('Hackathon_data/Hackathon_data/sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [3]:
len(sequence_wt)

656

In [4]:

def get_mutated_sequence(mut, sequence_wt):
    '''
    Adds the specified mutation into the wild-type sequence.

    Params: 
        mut (str): The mutation to be applied.
        sequence_wt (str): The wild-type sequence to which the mutation will be applied.
    Returns:
        A deep copy of the mutated sequence string.
    '''
    # wt - wild type; pos - position; mt - mutation
    wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

    sequence = deepcopy(sequence_wt)

    return sequence[:pos] + mt + sequence[pos+1:]

In [5]:
df_train = pd.read_csv('Hackathon_data/Hackathon_data/train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [6]:
df_test = pd.read_csv('Hackathon_data/Hackathon_data/test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [7]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a Prediction Model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

### Hyperparameters

In [8]:
seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

### ProteinDataset
Convert amino acids to machine-readable data via embedding space used by ESM-2.

In [ ]:
def gen_emb_from_df(df, out_dir="esm_embeddings_variants", device="cuda:0", batch_size=8):
    '''
    Method to generate embeddings and store them in target output directory.
    '''
    os.makedirs(out_dir, exist_ok=True)

    # Each item is (name_for_file, sequence)
    data = [(m, s[:1000]) for m, s in df[["mutant", "sequence"]].drop_duplicates().values]
    print(f"Number of unique variants: {len(data)}")

    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    batch_converter = alphabet.get_batch_converter()
    model.to(device).eval().half() # Half-prec. to shorten runtime

    for i in tqdm(range(int(np.ceil(len(data) / batch_size)))):
        batch = data[i * batch_size:(i + 1) * batch_size]

        # Cache skip
        if all(os.path.exists(os.path.join(out_dir, f"{name}.pt")) for name, _ in batch):
            continue

        labels, strs, tokens = batch_converter(batch)
        lens = (tokens != alphabet.padding_idx).sum(1)
        tokens = tokens.to(device)

        with torch.no_grad(), torch.autocast(device_type="cuda"):
            out = model(tokens, repr_layers=[33])

        reps = out["representations"][33].detach().cpu()

        for k, L in enumerate(lens):
            name = batch[k][0]
            path = os.path.join(out_dir, f"{name}.pt")
            if os.path.exists(path):
                continue

            # exclude BOS/EOS tokens for cleaner mean pooling
            seq_tokens = reps[k, 1:L-1]
            seq_mean = seq_tokens.mean(0)
            torch.save(seq_mean, path)

class DmsESMDataset(Dataset):
    def __init__(self, df, emb_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.emb_dir = emb_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        mutant = self.df.loc[idx, "mutant"]
        emb = torch.load(os.path.join(self.emb_dir, f"{mutant}.pt")).float()
        if self.is_train:
            y = torch.tensor(self.df.loc[idx, "DMS_score"], dtype=torch.float32)
            return emb, y
        return emb, torch.tensor(0.0, dtype=torch.float32)

In [33]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# Run on GPU 0
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [ ]:
# Combine our datasets for embedding, them split them into train / test sets afterward. Each has mutated sequence variants stored as a separate .pt embedding.
all_df = pd.concat(
    [df_train[["mutant", "sequence"]], df_test[["mutant", "sequence"]]],
    ignore_index=True
).drop_duplicates("mutant")

gen_emb_from_df(all_df, out_dir="esm_embeddings_variants", device=device, batch_size=8)

Number of unique variants: 12464


  0%|          | 0/1558 [00:00<?, ?it/s]

In [ ]:
gen_emb('Hackathon_Data/Hackathon_data/sequence.fasta', out_dir='esm_embeddings_train')
#gen_emb('Hackathon_Data/Hackathon_data/', out_dir='esm_embeddings_test')

Number of sequences: 1


  0%|          | 0/1 [00:00<?, ?it/s]

In [38]:
# Use simple MLP model to predict from ESM-2 embeddings.
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=1280, output_dim=len(sequence_wt), hidden_dim=640, dropout_p=0.1):
        super(MLPClassifier, self).__init__()
        
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid() # Ensures fitness scores in the range (0,1).
        )

    def forward(self, x):
        return self.layers(x)

In [40]:
def train_model_esm(model, train_dataset, val_dataset, epochs=100, batch_size=256, lr=1e-3, patience=10, device='cuda:0'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Use MSE loss to handle bounded regression.
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_acc = 0
    patience_counter = 0
    best_ckpt = None

    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = None
            loss = None
            ######################################################################
            # TODO: backpropogation
            optimizer.zero_grad()
            outputs = model.forward(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            ######################################################################

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)

        train_acc = correct / total

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for sequences, labels in val_loader:
                ######################################################################
                # TODO: inference
                sequences, labels = sequences.to(device), labels.to(device)
                outputs = model(sequences)
                preds = torch.argmax(outputs, dim=1)
                ######################################################################
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total
        print(f'Epoch {epoch+1}: Train Loss={total_loss:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}')

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            best_ckpt = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    return model, best_ckpt

In [41]:
# Save train and test datasets
train_ds = DmsESMDataset(df_train, "esm_embeddings_variants", is_train=True)
test_ds  = DmsESMDataset(df_test,  "esm_embeddings_variants", is_train=False)

# Split validation set
train_ds, val_ds = train_test_split(train_ds, test_size=val_ratio, random_state=seed, shuffle=True)

test_loader = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

In [42]:
# --------------- Train our model ---------------
model_esm = MLPClassifier(input_dim=1280, 
                          output_dim=len(sequence_wt), 
                          hidden_dim=640, 
                          dropout_p=0.3).to(device)
# TODO Currently using test values to make sure this works.
model_esm, best_ckpt_esm = train_model_esm(
    model_esm, 
    train_ds, 
    val_ds, 
    epochs=10, # 500
    batch_size=256, 
    lr=1e-3, # 1e-4
    patience=5, # 50, 
    device=device
)
model_esm.load_state_dict(best_ckpt_esm)


# --------------- Test our model ---------------
model_esm.eval()
preds = []
with torch.no_grad():
    for sequences, _ in test_loader:
        # Inference on the test set
        sequences = sequences.to(device)
        outputs = model_esm(sequences)
        preds.extend(torch.argmax(outputs, dim=1).cpu().tolist())


preds = [label_list[pred] for pred in preds]
df_preds = pd.DataFrame(preds)
df_preds.to_csv('test_predictions.csv', index=False, header=False)

c:\Users\Cole\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\nn\modules\loss.py:626: UserWarning: Using a target size (torch.Size([256])) that is different to the input size (torch.Size([256, 656])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (656) must match the size of tensor b (256) at non-singleton dimension 1

In [12]:
df_test['DMS_score_predicted'] = y_test_pred
df_test

,mutant,sequence,DMS_score_predicted
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
...,...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188


In [13]:
df_test[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv')

## 3. Select Query for Next Round

In [14]:
df_test.sort_values('DMS_score_predicted', ascending=False).head(100)

,mutant,sequence,DMS_score_predicted
28,N2A,MVAEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223190
265,L14A,MVNEARGNSSLNPCAEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223190
226,P12A,MVNEARGNSSLNACLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223190
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189
2282,M127A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223189
...,...,...,...
44,E3I,MVNIARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
45,E3L,MVNLARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
47,E3N,MVNNARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188
48,E3M,MVNMARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.223188


In [15]:
# Example: randomly select 100 test variants to be queried.
# Note: Random selection may not be a good strategy
# TODO: Select query mutants for the next round based on your own criteria

queries = np.random.choice(df_test.mutant.values, size=100, replace=False)
queries

array(['A654N', 'Y402I', 'V116T', 'P655W', 'L313G', 'K150Y', 'W58S',
       'K448L', 'A352V', 'F223E', 'I562K', 'M245N', 'E358A', 'K530Y',
       'K467C', 'T450K', 'D91E', 'G260C', 'L346F', 'N488V', 'D624I',
       'E231M', 'D176G', 'L641A', 'N643E', 'K509F', 'P342G', 'N443K',
       'P394V', 'K596Y', 'A523I', 'P159E', 'L313Q', 'S183A', 'F581V',
       'S205L', 'D628G', 'E359K', 'K530A', 'G35N', 'I283N', 'E382A',
       'D80W', 'P201T', 'R45L', 'R294G', 'P555V', 'R534V', 'S24N',
       'C451T', 'V449D', 'V239C', 'C192E', 'L75V', 'G85W', 'L134V',
       'F581Q', 'F236P', 'C249M', 'N556Y', 'G246R', 'P491D', 'R156N',
       'C451R', 'S439L', 'T339H', 'K26M', 'A219L', 'M153A', 'G544E',
       'W594A', 'Y531F', 'Q484E', 'C630E', 'N474R', 'L213N', 'E651L',
       'G169V', 'L313R', 'W164E', 'G101H', 'G503Y', 'N488H', 'S24K',
       'P453N', 'E23C', 'V361C', 'W420H', 'A349N', 'I611G', 'L270G',
       'L424Q', 'V618A', 'A550F', 'D401I', 'E513F', 'W632R', 'V535T',
       'L317F', 'K104D'], dtype

In [16]:
with open('query.txt', 'w') as f:
  for mutant in queries:
    f.write(mutant+'\n')